In [1]:
import os
from pypdf import PdfReader

In [2]:
pdf_path = "C:/Ijjat me learning/RAG Project/documents/machine_learning.pdf"

reader=PdfReader(pdf_path)

print("Total Pages : ", len(reader.pages))

Total Pages :  6


In [3]:
all_text=" "

for page_no, page in enumerate(reader.pages):
    text= page.extract_text()

    if text:
        all_text += text + "\n"

    print(f"Page {page_no + 1} extracted")


Page 1 extracted
Page 2 extracted
Page 3 extracted
Page 4 extracted
Page 5 extracted
Page 6 extracted


In [4]:
print(all_text)

 Learning notes  Beginner to intermediate
Page 1
 Machine Learning
 A practical foundation: concepts, workflow, algorithms and evaluation
How to use this guide
Read one section at a time. Reproduce the examples yourself, write down what you do not understand, and complete
the exercises without looking at the answers. Understanding the reasoning matters more than memorising definitions.
Learning outcomes

Explain supervised, unsupervised and reinforcement learning.

Distinguish classification from regression.

Prepare a dataset without leaking test information.

Train a baseline model and evaluate it with suitable metrics.

Recognise overfitting and explain ways to reduce it.
Prerequisites
Basic Python is helpful. You should be comfortable with variables, functions, lists and importing packages. Basic
algebra and averages are enough to begin; learn probability and statistics alongside the guide.

Learning notes  Beginner to intermediate
Page 2
1. What is machine learning?
Machine

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)

In [6]:
chunks=text_splitter.split_text(all_text)

print("Total chunks: ", len(chunks))

Total chunks:  27


In [7]:
for i, chunk in enumerate(chunks[:5]):
    print(f"\n--- Chunk {i+1} ---\n")
    print(chunk)


--- Chunk 1 ---

Learning notes  Beginner to intermediate
Page 1
 Machine Learning
 A practical foundation: concepts, workflow, algorithms and evaluation
How to use this guide
Read one section at a time. Reproduce the examples yourself, write down what you do not understand, and complete
the exercises without looking at the answers. Understanding the reasoning matters more than memorising definitions.
Learning outcomes

Explain supervised, unsupervised and reinforcement learning.


--- Chunk 2 ---

Learning outcomes

Explain supervised, unsupervised and reinforcement learning.

Distinguish classification from regression.

Prepare a dataset without leaking test information.

Train a baseline model and evaluate it with suitable metrics.

Recognise overfitting and explain ways to reduce it.
Prerequisites
Basic Python is helpful. You should be comfortable with variables, functions, lists and importing packages. Basic

--- Chunk 3 ---

algebra and averages are enough to begin; lear

In [8]:
from sentence_transformers import SentenceTransformer

model=SentenceTransformer("all-MiniLM-L6-v2")

chunk_embedding = model.encode(
    chunks,
    show_progress_bar=True
)

print("Number of chunks: ", len(chunks))
print("Embedding shape:", chunk_embedding.shape)

c:\Ijjat me learning\RAG Project\rag_project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 1/1 [00:01<00:00,  1.37s/it]

Number of chunks:  27
Embedding shape: (27, 384)


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(
    [chunk_embedding[0]],
    [chunk_embedding[1]]
)

print(similarity[0][0])

0.61570585


In [10]:
print(cosine_similarity(
    [chunk_embedding[0]],
    [chunk_embedding[2]]
)[0][0])

0.44167385


In [11]:
%pip install -U chromadb

Note: you may need to restart the kernel to use updated packages.


c:\Ijjat me learning\RAG Project\rag_project\Scripts\python.exe: No module named pip


In [12]:
import chromadb

client = chromadb.PersistentClient(
    path="./chroma_db"
)

collection = client.get_or_create_collection(
    name="rag_documents",
    metadata={"hnsw:space": "cosine"}
)

In [13]:
ids = [f"chunk_{i}" for i in range(len(chunks))]

collection.add(
    ids=ids,
    documents=chunks,
    embeddings=chunk_embedding.tolist()
)

In [14]:
collection_demo = client.get_or_create_collection(
    name="metadata_demo"
)

collection_demo.add(
    ids=["doc_1"],
    documents=["Machine learning is a branch of AI."],
    embeddings=[model.encode(
        "Machine learning is a branch of AI."
    ).tolist()],
    metadatas=[
        {
            "source": "machine_learning.pdf",
            "category": "Machine Learning"
        }
    ]
)

In [15]:
print("Total records:", collection.count())

Total records: 27


In [16]:
result = collection.get(
    limit=3,
    include=["documents", "metadatas"]
)

print(result)

{'ids': ['chunk_0', 'chunk_1', 'chunk_2'], 'embeddings': None, 'documents': ['Learning notes \x7f Beginner to intermediate\nPage 1\n Machine Learning\n A practical foundation: concepts, workflow, algorithms and evaluation\nHow to use this guide\nRead one section at a time. Reproduce the examples yourself, write down what you do not understand, and complete\nthe exercises without looking at the answers. Understanding the reasoning matters more than memorising definitions.\nLearning outcomes\n\x7f\nExplain supervised, unsupervised and reinforcement learning.\n\x7f', 'Learning outcomes\n\x7f\nExplain supervised, unsupervised and reinforcement learning.\n\x7f\nDistinguish classification from regression.\n\x7f\nPrepare a dataset without leaking test information.\n\x7f\nTrain a baseline model and evaluate it with suitable metrics.\n\x7f\nRecognise overfitting and explain ways to reduce it.\nPrerequisites\nBasic Python is helpful. You should be comfortable with variables, functions, lists and

In [17]:
query = "What is machine learning?"

query_embedding = model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

In [18]:
for i, document in enumerate(results["documents"][0]):
    print(f"\nResult {i+1}:")
    print(document)


Result 1:
Learning notes  Beginner to intermediate
Page 2
1. What is machine learning?
Machine learning (ML) is a field of computer science in which a system learns patterns from data and uses those
patterns to make predictions or decisions. Instead of writing every rule by hand, we provide examples and a
learning algorithm.
Example: rather than manually writing rules for every type of spam email, train a model on emails labelled spam or

Result 2:
examples. Performance depends on data quality, assumptions and evaluation.
2. Main types of machine learning
Supervised learning
The training data contains input features and a target label/value. The model learns a mapping from inputs to
targets.

Classification predicts a category: spam/not spam, churn/no churn, disease class.

Regression predicts a numeric value: house price, delivery time, sales.
Unsupervised learning

Result 3:
not spam. The model learns statistical patterns that help classify new messages.
Traditional programming v

In [19]:
questions = [
    "What is machine learning?",
    "Explain neural networks.",
    "What is generative AI?",
    "How does deep learning work?",
    "What is supervised learning?"
]

for question in questions:

    print("\nQUESTION:", question)

    query_embedding = model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    for i, document in enumerate(results["documents"][0]):
        print(f"\nResult {i+1}:")
        print(document[:300])


QUESTION: What is machine learning?

Result 1:
Learning notes  Beginner to intermediate
Page 2
1. What is machine learning?
Machine learning (ML) is a field of computer science in which a system learns patterns from data and uses those
patterns to make predictions or decisions. Instead of writing every rule by hand, we provide examples and a
le

Result 2:
examples. Performance depends on data quality, assumptions and evaluation.
2. Main types of machine learning
Supervised learning
The training data contains input features and a target label/value. The model learns a mapping from inputs to
targets.

Classification predicts a category: spam/not spam,

Result 3:
not spam. The model learns statistical patterns that help classify new messages.
Traditional programming vs machine learning
Traditional programming
Machine learning
Rules + input → output
Examples + expected outputs → learned model
A developer specifies decision logic
An algorithm estimates useful 

QUESTION: Explain neural n


Result 1:
Learning notes  Beginner to intermediate
Page 1
 Machine Learning
 A practical foundation: concepts, workflow, algorithms and evaluation
How to use this guide
Read one section at a time. Reproduce the examples yourself, write down what you do not understand, and complete
the exercises without looki

Result 2:
Learning notes  Beginner to intermediate
Page 2
1. What is machine learning?
Machine learning (ML) is a field of computer science in which a system learns patterns from data and uses those
patterns to make predictions or decisions. Instead of writing every rule by hand, we provide examples and a
le

Result 3:
examples. Performance depends on data quality, assumptions and evaluation.
2. Main types of machine learning
Supervised learning
The training data contains input features and a target label/value. The model learns a mapping from inputs to
targets.

Classification predicts a category: spam/not spam,

QUESTION: What is generative AI?

Result 1:
Learning notes  Be

In [20]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    where={"source": "machine_learning.pdf"}
)

In [21]:
%pip install -U groq python-dotenv

Note: you may need to restart the kernel to use updated packages.


c:\Ijjat me learning\RAG Project\rag_project\Scripts\python.exe: No module named pip


In [22]:
from dotenv import load_dotenv
from groq import Groq
import os

load_dotenv()
client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

print("GROQ client initialized")

GROQ client initialized


In [24]:
response=client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Explain machine learning in simple words."
        }
    ],
    temperature=0.2
)

print(response.choices[0].message.content)

**Machine learning is like teaching a computer to learn from experience instead of being told exactly what to do.**

1. **Give it examples** – You show the computer many pictures of cats and dogs, or many emails that are spam or not spam.
2. **It looks for patterns** – The computer scans those examples and notices things that are common in cats (pointy ears, whiskers) and different in dogs (longer snouts, floppy ears).
3. **It builds a rule** – From those patterns it creates a “rule” or a model that says, “If the picture has these features, it’s probably a cat.”
4. **Test it** – You give it a new picture it hasn’t seen before. If the rule works, it will correctly guess “cat” or “dog.”
5. **Improve** – If it makes mistakes, you give it more examples or tweak the rule, and it gets better over time.

So, machine learning is simply a way for computers to learn from data, spot patterns, and make predictions or decisions without being explicitly programmed for every single case.


In [ ]:
def retrieve_context(question, top_k=3):
   
    #Convert the question into an embedding
    query_embedding = model.encode(question).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    retrieved_documents = results["documents"][0]

    return retrieved_documents

In [29]:
question = "What is machine learning?"

context = retrieve_context(question)

for i, doc in enumerate(context):
    print(f"\nChunk {i+1}:")
    print(doc)


Chunk 1:
Learning notes  Beginner to intermediate
Page 2
1. What is machine learning?
Machine learning (ML) is a field of computer science in which a system learns patterns from data and uses those
patterns to make predictions or decisions. Instead of writing every rule by hand, we provide examples and a
learning algorithm.
Example: rather than manually writing rules for every type of spam email, train a model on emails labelled spam or

Chunk 2:
examples. Performance depends on data quality, assumptions and evaluation.
2. Main types of machine learning
Supervised learning
The training data contains input features and a target label/value. The model learns a mapping from inputs to
targets.

Classification predicts a category: spam/not spam, churn/no churn, disease class.

Regression predicts a numeric value: house price, delivery time, sales.
Unsupervised learning

Chunk 3:
not spam. The model learns statistical patterns that help classify new messages.
Traditional programming vs m

In [30]:
def create_prompt(question, context):

    context_text="\n\n".join(context)

    prompt = f"""
    You are an AI assistant that answers questions
    using the provided context.

    Rules:
    1. Answer only using the provided context.
    2. Do not make up information.
    3. If the answer is not available in the context,
       say "I could not find this information in the documents."

    Context:
    {context_text}

    Question:
    {question}

    Answer:
    """

    return prompt

In [31]:
def generate_answer(question, context):

    prompt=create_prompt(question, context)

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [36]:
def rag_pipeline(question):
    # Retrieve relevant chunks
    context=retrieve_context(question)

    # Generate answer using retrieved context
    answer = generate_answer(question, context)

    return answer

In [39]:
question="How to play chess?"

answer=rag_pipeline(question)

print("Question: ", question)
print("Answer: ", answer)

Question:  How to play chess?
Answer:  I could not find this information in the documents.
